In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Uniform sampling from the 2D annulus: 2 <= radius <= 3
rng = np.random.default_rng(42)
n_points = 5_000
inner_radius = 2.0
outer_radius = 3.0

theta = rng.uniform(0.0, 2.0 * np.pi, n_points)
r = np.sqrt(rng.uniform(inner_radius**2, outer_radius**2, n_points))

x = r * np.cos(theta)
y = r * np.sin(theta)
points = np.column_stack((x, y))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.6, edgecolors='none')

inner_boundary = plt.Circle((0, 0), inner_radius, fill=False, linestyle='--', color='tab:red', linewidth=2, label='inner radius = 2')
outer_boundary = plt.Circle((0, 0), outer_radius, fill=False, color='black', linewidth=2, label='outer radius = 3')
ax.add_patch(inner_boundary)
ax.add_patch(outer_boundary)

ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-3.3, 3.3)
ax.set_ylim(-3.3, 3.3)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Uniform samples from an annulus')
ax.legend(loc='upper right')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gaussian datasets conditioned to lie on the annulus: 2 <= radius <= 3
rng = np.random.default_rng(123)
inner_radius = 2.0
outer_radius = 3.0
variance = 0.25
n_points_per_gaussian = 250

mean_left = np.array([-2.5, 0.0])
mean_right = np.array([2.5, 0.0])

def in_annulus(samples, inner_radius=2.0, outer_radius=3.0):
    radii = np.linalg.norm(samples, axis=1)
    return (radii >= inner_radius) & (radii <= outer_radius)

def sample_gaussian_on_annulus(mean, variance, n_samples, rng, inner_radius=2.0, outer_radius=3.0, batch_size=10_000):
    std = np.sqrt(variance)
    accepted = []
    n_drawn = 0
    n_accepted = 0

    while n_accepted < n_samples:
        candidates = rng.normal(loc=mean, scale=std, size=(batch_size, 2))
        n_drawn += len(candidates)
        kept = candidates[in_annulus(candidates, inner_radius, outer_radius)]

        if len(kept) > 0:
            accepted.append(kept)
            n_accepted += len(kept)

    return np.vstack(accepted)[:n_samples], n_drawn, n_accepted

left_gaussian_points, left_drawn, left_accepted = sample_gaussian_on_annulus(
    mean_left, variance, n_points_per_gaussian, rng, inner_radius, outer_radius
)
right_gaussian_points, right_drawn, right_accepted = sample_gaussian_on_annulus(
    mean_right, variance, n_points_per_gaussian, rng, inner_radius, outer_radius
)

gaussian_ring_points = np.vstack((left_gaussian_points, right_gaussian_points))
gaussian_ring_labels = np.concatenate((
    np.zeros(n_points_per_gaussian, dtype=int),
    np.ones(n_points_per_gaussian, dtype=int),
))

print(f'Left Gaussian retained {n_points_per_gaussian} points; acceptance rate in generated batches: {left_accepted / left_drawn:.2%}')
print(f'Right Gaussian retained {n_points_per_gaussian} points; acceptance rate in generated batches: {right_accepted / right_drawn:.2%}')

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(left_gaussian_points[:, 0], left_gaussian_points[:, 1], s=10, alpha=0.55, edgecolors='none', label='Gaussian centered at (-2.5, 0)')
ax.scatter(right_gaussian_points[:, 0], right_gaussian_points[:, 1], s=10, alpha=0.55, edgecolors='none', label='Gaussian centered at (2.5, 0)')
ax.scatter([mean_left[0], mean_right[0]], [mean_left[1], mean_right[1]], marker='x', s=120, linewidths=2.5, color='black', label='Gaussian centers')

inner_boundary = plt.Circle((0, 0), inner_radius, fill=False, linestyle='--', color='tab:red', linewidth=2, label='inner radius = 2')
outer_boundary = plt.Circle((0, 0), outer_radius, fill=False, color='black', linewidth=2, label='outer radius = 3')
ax.add_patch(inner_boundary)
ax.add_patch(outer_boundary)

ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-3.3, 3.3)
ax.set_ylim(-3.3, 3.3)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gaussian samples retained on the annulus')
ax.legend(loc='upper right')
ax.grid(alpha=0.25)
plt.show()

## Construction data for the kernel-profile comparison

The uniform annulus samples define the shared construction set for all five kernels. The Gaussian clouds provide starting points and visual context. Each profile is fitted on the same symmetric CkNN support, then supplies a Laplacian, biharmonic spectral embedding, continuous Nyström extension, and RFM vector field. The experiment compares raw and boundary-guarded trajectories and checks their behavior when the Euler step size is halved.

In [ ]:
all_generated_points = np.vstack((points, gaussian_ring_points))
construction_points = points.astype(np.float32)

source_labels = np.concatenate(
    (
        np.full(points.shape[0], "uniform annulus", dtype=object),
        np.where(gaussian_ring_labels == 0, "left Gaussian", "right Gaussian").astype(object),
    )
)

source_palette = {
    "uniform annulus": "tab:gray",
    "left Gaussian": "tab:blue",
    "right Gaussian": "tab:orange",
}

print("all_generated_points shape:", all_generated_points.shape)
print("construction_points shape:", construction_points.shape)
print("construction source: uniform annulus samples only")
print("source counts:")
for label in source_palette:
    print(f"  {label}: {np.sum(source_labels == label)}")

fig, ax = plt.subplots(figsize=(7, 6))
for label, color in source_palette.items():
    mask = source_labels == label
    ax.scatter(
        all_generated_points[mask, 0],
        all_generated_points[mask, 1],
        s=8,
        alpha=0.45,
        edgecolors="none",
        color=color,
        label=label,
    )

ax.scatter(
    construction_points[:, 0],
    construction_points[:, 1],
    s=10,
    alpha=0.18,
    edgecolors="none",
    color="black",
    label="used for kernel/Laplacian construction",
)
ax.add_patch(
    plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red", linewidth=2)
)
ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, color="black", linewidth=2))
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-3.3, 3.3)
ax.set_ylim(-3.3, 3.3)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Generated datapoints and uniform-only construction set")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
plt.show()

### Shared annulus projection and boundary helpers

Every profile uses the same annulus geometry and boundary probes. The point projection clips radii to the interval [2, 3]. The annulus is two-dimensional in its two-dimensional ambient space, so the interior tangent-space projection is the identity. At a boundary, the tangent-cone guard removes outward velocity while retaining inward and tangential components. The raw trajectories evaluate the analytical field directly; the guarded trajectories use this common cone guard and project after every Euler step.

In [ ]:
import jax.numpy as jnp
from cellflow.ode_solver._rfm_interpolation import RFMInterpolation

def project_to_annulus(x, inner=inner_radius, outer=outer_radius, eps=1e-12):
    x = jnp.asarray(x)
    radius = jnp.linalg.norm(x, axis=-1, keepdims=True)
    clipped_radius = jnp.clip(radius, inner, outer)
    fallback_direction = jnp.zeros_like(x).at[..., 0].set(1.0)
    direction = jnp.where(radius > eps, x / jnp.maximum(radius, eps), fallback_direction)
    return direction * clipped_radius


def annulus_interior_normal(x, inner=inner_radius, outer=outer_radius, boundary_tol=1e-4, eps=1e-12):
    y = project_to_annulus(x, inner=inner, outer=outer, eps=eps)
    radius = jnp.linalg.norm(y, axis=-1, keepdims=True)
    radial = y / jnp.maximum(radius, eps)

    inner_mask = radius <= inner + boundary_tol
    outer_mask = radius >= outer - boundary_tol
    boundary_mask = inner_mask | outer_mask
    normal = jnp.where(inner_mask, radial, jnp.where(outer_mask, -radial, jnp.zeros_like(radial)))
    return normal, boundary_mask


def project_annulus_velocity_to_tangent_cone(x, v, boundary_tol=1e-4):
    v = jnp.asarray(v)
    normal, boundary_mask = annulus_interior_normal(x, boundary_tol=boundary_tol)
    normal_component = jnp.sum(v * normal, axis=-1, keepdims=True)
    outward_mask = boundary_mask & (normal_component < 0.0)
    return jnp.where(outward_mask, v - normal_component * normal, v)


class BoundarySafeAnnulusRFMInterpolation(RFMInterpolation):
    def u_t(self, xt, x1, t):
        y = project_to_annulus(xt)
        v = super().u_t(y, x1, t)
        return project_annulus_velocity_to_tangent_cone(y, v)


def sample_annulus_boundary(n_angles=96):
    angles = np.linspace(0.0, 2.0 * np.pi, n_angles, endpoint=False, dtype=np.float32)
    radial = np.column_stack((np.cos(angles), np.sin(angles))).astype(np.float32)
    inner_points = (inner_radius * radial).astype(np.float32)
    outer_points = (outer_radius * radial).astype(np.float32)
    points_boundary = np.vstack([inner_points, outer_points]).astype(np.float32)
    interior_normals = np.vstack([radial, -radial]).astype(np.float32)
    return points_boundary, interior_normals


boundary_points, boundary_normals = sample_annulus_boundary()


## Symmetric variable-bandwidth kernels: fields and trajectories

This comparison uses the same uniform construction points and CkNN support for every profile:
$\rho_i=\|x_i-x_{i,k_\rho}\|$, $s_{ij}=\|x_i-x_j\|^2/(\delta^2\rho_i\rho_j)$, with edges where $s_{ij}<1$. We compare a fixed-bandwidth Gaussian on this support, a self-tuning Gaussian, $(1-s)_+^4$, a smooth bump, and binary CkNN. Gaussian profiles are truncated here; setting compact_support=False also implements the full Gaussian reference.

| Profile | Weight on a CkNN edge ($s<1$) |
| --- | --- |
| Fixed Gaussian | $\exp(-\|x-y\|^2/(2\sigma^2))$ |
| Variable Gaussian | $\exp(-s/2)$ |
| Compact polynomial | $(1-s)^4$ |
| Smooth bump | $\exp(1-1/(1-s))$ |
| Binary CkNN | $1$ |

The shared support uses kernel_density_k=24 and kernel_delta=2.0. All profiles share construction points, edge candidates, spectral weighting, field probes, starting points, targets, and integration steps. The fixed Gaussian bandwidth is the median positive distance across the 24 non-self nearest neighbors of the construction points. Smooth profiles can lose extremely small weights to underflow; the table reports their retained edges.

By default we use $L=D-W$, matching [Berry and Sauer, Section 6](https://math.gmu.edu/~berry/Publications/CkNN.pdf#page=23). Set kernel_laplacian_type="normalized" to compare the normalized Laplacian. The Nyström formulas, including spectral scales $a_\ell$, are

$$
\Psi_\ell(x)=
\begin{cases}
a_\ell\displaystyle\frac{\sum_jw_j(x)\phi_{j\ell}}{d(x)-\lambda_\ell}, & L=D-W,\\[8pt]
a_\ell\displaystyle\frac{\sum_jw_j(x)\phi_{j\ell}/\sqrt{d_j}}{\sqrt{d(x)}(1-\lambda_\ell)}, & L=I-D^{-1/2}WD^{-1/2}.
\end{cases}
$$

Analytic gradients differentiate both $d(x)$ and $\rho(x)$. Query scales use the $(k_\rho+1)$th training point, including a possible self-match, so they continuously reproduce the fitted non-self scales at training coordinates. This scale estimator is piecewise smooth at neighbor ties. Normalized graphs retain self-affinities for agreement with the continuous extension; self-affinities cancel from $D-W$.

Binary CkNN has zero extension gradients almost everywhere and therefore produces stationary RFM trajectories. Truncated Gaussians have support-boundary jumps. The polynomial and bump remove those jumps, while retaining possible derivative changes from the bandwidth estimator. Numerical derivative checks and training-coordinate reconstruction are reported before the flow comparison.

In [ ]:
import time

import jax
import jax.numpy as jnp
import pandas as pd
from IPython.display import display
from scipy.sparse.csgraph import connected_components
from scipy.sparse.linalg import eigsh

from cellflow.preprocessing._kernels import SymmetricBandwidthKernel
from cellflow.preprocessing._premetric import (
    SpectralNystroem,
    normalized_laplacian,
    spectral_embedding_scale,
    unnormalized_laplacian,
)

kernel_density_k = 24
kernel_delta = 2.0
kernel_n_modes = 32
kernel_laplacian_type = "unnormalized"
kernel_spectral_weight_type = "biharmonic"
kernel_profiles = {
    "fixed Gaussian": "fixed_gaussian",
    "variable Gaussian": "gaussian",
    "compact polynomial": "polynomial",
    "smooth bump": "bump",
    "binary CkNN": "indicator",
}
kernel_models = {}
kernel_graphs = {}
kernel_fitted = {}
kernel_fit_rows = []

for kernel_name, profile in kernel_profiles.items():
    fit_started = time.perf_counter()
    fitted_kernel = SymmetricBandwidthKernel(
        construction_points,
        profile=profile,
        density_k=kernel_density_k,
        delta=kernel_delta,
    )
    kernel_graph = fitted_kernel.graph(
        include_self=kernel_laplacian_type == "normalized"
    ).astype(np.float64)
    component_count = connected_components(kernel_graph, directed=False)[0]
    if component_count != 1:
        raise ValueError(f"{kernel_name}: {component_count} components; increase density_k or delta.")
    laplacian = (
        unnormalized_laplacian(kernel_graph)
        if kernel_laplacian_type == "unnormalized"
        else normalized_laplacian(kernel_graph)
    )
    eigenvalues, eigenvectors = eigsh(
        laplacian, k=kernel_n_modes + 1, sigma=-1e-8, which="LM", tol=1e-9,
        v0=np.random.default_rng(0).standard_normal(len(construction_points)),
    )
    eigen_order = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[eigen_order][1:]
    eigenvectors = eigenvectors[:, eigen_order][:, 1:]
    component_scales = spectral_embedding_scale(
        eigenvalues, weight_type=kernel_spectral_weight_type
    )
    model = SpectralNystroem(
        construction_points,
        eigenvectors,
        eigenvalues,
        sigma=None,
        psi=eigenvectors * component_scales,
        x_to_id=construction_points,
        graph=kernel_graph,
        kernel=fitted_kernel,
        laplacian_type=kernel_laplacian_type,
        spectral_scale=component_scales,
        target_embedding="nystroem",
    )
    reconstruction_ids = np.linspace(0, len(construction_points) - 1, 20, dtype=int)
    reconstruction = np.asarray(jax.jit(jax.vmap(model.extend_point))(
        jnp.asarray(construction_points[reconstruction_ids])
    ))
    expected_coordinates = np.asarray(model.Psi)[reconstruction_ids]
    reconstruction_error = np.linalg.norm(reconstruction - expected_coordinates) / max(
        np.linalg.norm(expected_coordinates), 1e-12
    )
    kernel_models[kernel_name] = model
    kernel_graphs[kernel_name] = kernel_graph
    kernel_fitted[kernel_name] = fitted_kernel
    degree_values = np.asarray(kernel_graph.sum(axis=1)).ravel()
    kernel_fit_rows.append({
        "kernel": kernel_name,
        "edges": (kernel_graph.nnz - np.count_nonzero(kernel_graph.diagonal())) // 2,
        "components": component_count,
        "min_degree": degree_values.min(),
        "median_degree": np.median(degree_values),
        "lambda_2": eigenvalues[0],
        "max_eigen_residual": np.max(np.linalg.norm(
            laplacian @ eigenvectors - eigenvectors * eigenvalues, axis=0
        )),
        "extension_relative_error": reconstruction_error,
        "fit_seconds": time.perf_counter() - fit_started,
    })
    print(f"{kernel_name}: fitted in {kernel_fit_rows[-1]['fit_seconds']:.2f}s", flush=True)

kernel_fit_df = pd.DataFrame(kernel_fit_rows).set_index("kernel")
kernel_fit_df["retained_edge_fraction"] = (
    kernel_fit_df["edges"] / kernel_fit_df.loc["binary CkNN", "edges"]
)
display(kernel_fit_df)

In [ ]:
kernel_field_angles = np.linspace(0.0, 2.0 * np.pi, 30, endpoint=False)
kernel_field_radii = np.linspace(inner_radius + 0.07, outer_radius - 0.07, 6)
kernel_angle_grid, kernel_radius_grid = np.meshgrid(
    kernel_field_angles, kernel_field_radii
)
kernel_field_points = np.column_stack((
    (kernel_radius_grid * np.cos(kernel_angle_grid)).ravel(),
    (kernel_radius_grid * np.sin(kernel_angle_grid)).ravel(),
)).astype(np.float32)
kernel_target = np.array([2.5, 0.0], dtype=np.float32)
kernel_field_targets = np.broadcast_to(kernel_target, kernel_field_points.shape)
kernel_field_time = 0.5
kernel_fields = {}
kernel_field_rows = []
kernel_raw_interpolators = {}

for kernel_name, model in kernel_models.items():
    interpolator = RFMInterpolation(model, project_to_manifold=False)
    kernel_raw_interpolators[kernel_name] = interpolator
    velocities = np.asarray(jax.jit(interpolator.u_t)(
        jnp.asarray(kernel_field_points), jnp.asarray(kernel_field_targets), kernel_field_time
    ))
    if not np.all(np.isfinite(velocities)):
        raise ValueError(f"Non-finite vector field for {kernel_name}.")
    speeds = np.linalg.norm(velocities, axis=1)
    support_counts = np.asarray(jax.jit(jax.vmap(
        lambda point: jnp.count_nonzero(kernel_fitted[kernel_name](point))
    ))(jnp.asarray(kernel_field_points)))
    with jax.enable_x64():
        gradient_query = jnp.array([2.44, 0.27], dtype=jnp.float64)
        gradient_target = jnp.asarray(kernel_target, dtype=jnp.float64)

        def checked_distance(point):
            residual = model.extend_point(point) - model.extend_point(gradient_target)
            return jnp.sqrt(jnp.dot(residual, residual) + model.eps)

        _, analytic_gradient, _ = model.distance_grad_and_norm(
            gradient_query, gradient_target
        )
        autodiff_gradient = jax.grad(checked_distance)(gradient_query)
        difference_step = 1e-6
        finite_difference_gradient = np.array([
            (float(checked_distance(gradient_query + direction))
             - float(checked_distance(gradient_query - direction))) / (2.0 * difference_step)
            for direction in np.eye(2) * difference_step
        ])
        gradient_scale = max(float(jnp.linalg.norm(analytic_gradient)), 1e-10)
        autodiff_error = float(jnp.linalg.norm(
            analytic_gradient - autodiff_gradient
        )) / gradient_scale
        finite_difference_error = float(np.linalg.norm(
            np.asarray(analytic_gradient) - finite_difference_gradient
        )) / gradient_scale
    if autodiff_error > 1e-5:
        raise AssertionError(f"{kernel_name}: analytic/autodiff gradient mismatch.")
    boundary_velocities = np.asarray(jax.jit(interpolator.u_t)(
        jnp.asarray(boundary_points),
        jnp.broadcast_to(jnp.asarray(kernel_target), boundary_points.shape),
        kernel_field_time,
    ))
    kernel_fields[kernel_name] = velocities
    kernel_field_rows.append({
        "kernel": kernel_name,
        "median_speed": np.median(speeds),
        "p95_speed": np.quantile(speeds, 0.95),
        "zero_speed_fraction": np.mean(speeds < 1e-8),
        "unsupported_probe_fraction": np.mean(support_counts == 0),
        "outward_boundary_fraction": np.mean(
            np.sum(boundary_velocities * boundary_normals, axis=1) < -1e-6
        ),
        "autodiff_relative_error": autodiff_error,
        "finite_difference_relative_error": finite_difference_error,
    })

kernel_field_df = pd.DataFrame(kernel_field_rows).set_index("kernel")
display(kernel_field_df)

def style_kernel_annulus(axis):
    axis.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    axis.add_patch(plt.Circle((0, 0), outer_radius, fill=False, color="black"))
    axis.scatter(*kernel_target, s=100, marker="*", color="black", zorder=5)
    axis.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), xlabel="x", ylabel="y")
    axis.set_aspect("equal", adjustable="box")
    axis.grid(alpha=0.15)

kernel_speed_colors = np.concatenate([
    np.log1p(np.linalg.norm(velocities, axis=1))
    for velocities in kernel_fields.values()
])
kernel_color_max = max(float(np.quantile(kernel_speed_colors, 0.98)), 1e-6)
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
for axis, (kernel_name, velocities) in zip(axes.ravel(), kernel_fields.items()):
    speeds = np.linalg.norm(velocities, axis=1)
    directions = 0.18 * velocities / np.maximum(speeds[:, None], 1e-8)
    quiver = axis.quiver(
        kernel_field_points[:, 0], kernel_field_points[:, 1],
        directions[:, 0], directions[:, 1], np.log1p(speeds),
        cmap="plasma", clim=(0.0, kernel_color_max),
        angles="xy", scale_units="xy", scale=1, width=0.005,
    )
    style_kernel_annulus(axis)
    axis.set_title(f"{kernel_name}\nzero velocities: {np.mean(speeds < 1e-8):.0%}")
for axis in axes.ravel()[len(kernel_fields):]:
    axis.set_visible(False)
fig.colorbar(quiver, ax=list(axes.ravel()), label="log(1 + raw speed), shared scale")
fig.suptitle("Raw RFM fields at t=0.5; arrows show direction")
plt.show()

In [ ]:
kernel_start_rng = np.random.default_rng(2026)
kernel_start_ids = kernel_start_rng.choice(len(left_gaussian_points), 12, replace=False)
kernel_starts = left_gaussian_points[kernel_start_ids].astype(np.float32)
kernel_final_time = 0.99
kernel_integration_steps = 800
kernel_step_times = np.linspace(
    0.0, kernel_final_time, kernel_integration_steps + 1, dtype=np.float32
)
kernel_trajectories = {}
kernel_distance_traces = {}
kernel_trajectory_rows = []

def integrate_kernel_paths(model, guarded, steps=None):
    integration_steps = kernel_integration_steps if steps is None else steps
    interpolator = (
        BoundarySafeAnnulusRFMInterpolation(
            model, projection_fn=project_to_annulus, project_to_manifold=False
        )
        if guarded else RFMInterpolation(model, project_to_manifold=False)
    )
    targets = jnp.broadcast_to(jnp.asarray(kernel_target), kernel_starts.shape)
    time_step = kernel_final_time / integration_steps
    integration_times = np.linspace(
        0.0, kernel_final_time, integration_steps + 1, dtype=np.float32
    )

    def integrate(starts):
        def step(state, current_time):
            velocities = interpolator.u_t(state, targets, current_time)
            proposed = state + time_step * velocities
            next_state = interpolator.project(proposed)
            return next_state, (next_state, proposed)

        _, (path, proposed_path) = jax.lax.scan(
            step, starts, jnp.asarray(integration_times[:-1])
        )
        return jnp.concatenate((starts[None], path), axis=0), proposed_path

    return jax.jit(integrate)(jnp.asarray(kernel_starts))

def kernel_path_geometry(path):
    segment_starts = path[:-1]
    segments = np.diff(path, axis=0)
    segment_lengths_sq = np.sum(segments**2, axis=-1)
    closest_fractions = np.clip(
        -np.sum(segment_starts * segments, axis=-1)
        / np.maximum(segment_lengths_sq, 1e-20), 0.0, 1.0
    )
    closest_points = segment_starts + closest_fractions[..., None] * segments
    closest_radii = np.linalg.norm(closest_points, axis=-1)
    radii = np.linalg.norm(path, axis=-1)
    return {
        "max_step_length": np.sqrt(segment_lengths_sq.max()),
        "max_segment_hole_penetration": np.maximum(inner_radius - closest_radii, 0.0).max(),
        "hole_crossing_segment_fraction": np.mean(closest_radii < inner_radius - 1e-3),
        "max_domain_violation": np.maximum(
            np.maximum(inner_radius - radii, 0.0),
            np.maximum(radii - outer_radius, 0.0),
        ).max(),
    }

for kernel_name, model in kernel_models.items():
    for mode, guarded in [("unprojected", False), ("annulus + cone", True)]:
        integration_started = time.perf_counter()
        path, proposed_path = integrate_kernel_paths(model, guarded)
        path = np.asarray(path)
        proposed_path = np.asarray(proposed_path)
        if not np.all(np.isfinite(path)):
            raise ValueError(f"Non-finite trajectories for {kernel_name}, {mode}.")
        sampled_path = path[::8]
        sampled_targets = np.broadcast_to(kernel_target, sampled_path.shape)
        spectral_distances = np.asarray(jax.jit(lambda states: jax.lax.map(
            lambda batch: model.distance_grad_and_norm_batch(
                batch, jnp.asarray(sampled_targets[0])
            )[0], states
        ))(jnp.asarray(sampled_path)))
        radii = np.linalg.norm(path, axis=-1)
        violation = np.maximum(inner_radius - radii, 0.0) + np.maximum(radii - outer_radius, 0.0)
        proposed_radii = np.linalg.norm(proposed_path, axis=-1)
        proposed_violation = (
            np.maximum(inner_radius - proposed_radii, 0.0)
            + np.maximum(proposed_radii - outer_radius, 0.0)
        )
        final_errors = np.linalg.norm(path[-1] - kernel_target, axis=1)
        initial_errors = np.linalg.norm(path[0] - kernel_target, axis=1)
        path_lengths = np.linalg.norm(np.diff(path, axis=0), axis=-1).sum(axis=0)
        unsupported_final = np.asarray(jax.jit(jax.vmap(
            lambda point: jnp.count_nonzero(kernel_fitted[kernel_name](point)) == 0
        ))(jnp.asarray(path[-1])))
        kernel_trajectories[kernel_name, mode] = path
        kernel_distance_traces[kernel_name, mode] = spectral_distances
        kernel_trajectory_rows.append({
            "kernel": kernel_name,
            "mode": mode,
            "mean_final_euclidean": np.mean(final_errors),
            "p90_final_euclidean": np.quantile(final_errors, 0.9),
            "mean_relative_euclidean": np.mean(final_errors / initial_errors),
            "fraction_within_0.1": np.mean(final_errors < 0.1),
            "mean_final_relative_spectral": np.mean(
                spectral_distances[-1] / np.maximum(spectral_distances[0], 1e-12)
            ),
            "mean_path_length": np.mean(path_lengths),
            "max_annulus_violation": np.max(violation),
            "max_proposed_violation": np.max(proposed_violation),
            "unsupported_final_fraction": np.mean(unsupported_final),
            "integration_seconds": time.perf_counter() - integration_started,
            **kernel_path_geometry(path),
        })
        print(
            f"{kernel_name}, {mode}: mean endpoint error {np.mean(final_errors):.4f}, "
            f"{np.mean(final_errors < 0.1):.0%} within 0.1; "
            f"{kernel_trajectory_rows[-1]['integration_seconds']:.2f}s",
            flush=True,
        )

kernel_trajectory_df = pd.DataFrame(kernel_trajectory_rows)
display(kernel_trajectory_df.sort_values(["mode", "mean_final_euclidean"]).reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(
    2, len(kernel_models), figsize=(25, 10), sharex=True, sharey=True,
    constrained_layout=True,
)
for column, kernel_name in enumerate(kernel_models):
    for row, mode in enumerate(["unprojected", "annulus + cone"]):
        axis = axes[row, column]
        path = kernel_trajectories[kernel_name, mode]
        for start_index in range(len(kernel_starts)):
            axis.plot(path[:, start_index, 0], path[:, start_index, 1], linewidth=1.1, alpha=0.7)
        axis.scatter(kernel_starts[:, 0], kernel_starts[:, 1], s=16, color="tab:blue")
        axis.scatter(path[-1, :, 0], path[-1, :, 1], s=18, color="tab:orange", marker="x")
        style_kernel_annulus(axis)
        result_row = kernel_trajectory_df[
            kernel_trajectory_df["kernel"].eq(kernel_name)
            & kernel_trajectory_df["mode"].eq(mode)
        ].iloc[0]
        axis.set_title(
            f"{kernel_name}: {mode}\n"
            f"mean endpoint error {result_row['mean_final_euclidean']:.3f}"
        )
fig.suptitle(
    f"Shared starts and target; blue starts, orange endpoints; {kernel_integration_steps} Euler steps"
)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
for column, mode in enumerate(["unprojected", "annulus + cone"]):
    for kernel_name in kernel_models:
        path = kernel_trajectories[kernel_name, mode]
        ambient_errors = np.linalg.norm(path - kernel_target, axis=-1)
        axes[0, column].plot(
            kernel_step_times, ambient_errors.mean(axis=1), label=kernel_name
        )
        distances = kernel_distance_traces[kernel_name, mode]
        axes[1, column].plot(
            kernel_step_times[::8],
            (distances / np.maximum(distances[0], 1e-12)).mean(axis=1),
            label=kernel_name,
        )
    axes[0, column].set(
        title=mode, xlabel="time", ylabel="mean Euclidean distance to target"
    )
    axes[1, column].plot(
        kernel_step_times, 1.0 - kernel_step_times, "k--", label="ideal 1 - t"
    )
    axes[1, column].set(
        title=mode, xlabel="time", ylabel="mean relative spectral distance"
    )
    axes[0, column].legend()
    axes[1, column].legend()
    axes[0, column].grid(alpha=0.2)
    axes[1, column].grid(alpha=0.2)
fig.suptitle("Endpoint convergence and spectral decay for every kernel")
plt.show()

guarded_kernel_results = kernel_trajectory_df[
    kernel_trajectory_df["mode"].eq("annulus + cone")
].sort_values("mean_final_euclidean")
valid_guarded_results = guarded_kernel_results[
    guarded_kernel_results["max_segment_hole_penetration"].le(1e-3)
    & guarded_kernel_results["max_domain_violation"].le(1e-3)
    & guarded_kernel_results["unsupported_final_fraction"].eq(0.0)
]
if len(valid_guarded_results):
    best_kernel_result = valid_guarded_results.iloc[0]
    print(
        f"Lowest endpoint error passing domain/segment checks: {best_kernel_result['kernel']} "
        f"({best_kernel_result['mean_final_euclidean']:.4f})."
    )
print(
    "Binary CkNN is a graph baseline: its almost-everywhere zero gradients "
    "leave the moving-point interpolation stationary."
)
print(
    "Annulus projection enforces the domain. Compare the raw boundary-field "
    "diagnostic and unprojected paths to assess the pre-metric itself."
)

In [ ]:
kernel_refinement_steps = 2 * kernel_integration_steps
kernel_refinement_rows = []
kernel_refined_trajectories = {}

for kernel_name, model in kernel_models.items():
    refinement_started = time.perf_counter()
    refined_path, _ = integrate_kernel_paths(
        model, guarded=True, steps=kernel_refinement_steps
    )
    refined_path = np.asarray(refined_path)
    if not np.all(np.isfinite(refined_path)):
        raise ValueError(f"{kernel_name}: non-finite refined trajectories.")
    refined_errors = np.linalg.norm(refined_path[-1] - kernel_target, axis=1)
    coarse_path = kernel_trajectories[kernel_name, "annulus + cone"]
    coarse_errors = np.linalg.norm(coarse_path[-1] - kernel_target, axis=1)
    geometry = kernel_path_geometry(refined_path)
    kernel_refined_trajectories[kernel_name] = refined_path
    kernel_refinement_rows.append({
        "kernel": kernel_name,
        "coarse_mean_endpoint_error": np.mean(coarse_errors),
        "refined_mean_endpoint_error": np.mean(refined_errors),
        "mean_endpoint_shift": np.linalg.norm(
            refined_path[-1] - coarse_path[-1], axis=1
        ).mean(),
        "refined_fraction_within_0.1": np.mean(refined_errors < 0.1),
        "passes_domain_segment_checks": (
            geometry["max_domain_violation"] <= 1e-3
            and geometry["max_segment_hole_penetration"] <= 1e-3
        ),
        **geometry,
        "refinement_seconds": time.perf_counter() - refinement_started,
    })
    print(
        f"{kernel_name}, {kernel_refinement_steps} guarded steps: "
        f"mean endpoint error {np.mean(refined_errors):.4f}; "
        f"maximum hole penetration {geometry['max_segment_hole_penetration']:.4f}",
        flush=True,
    )

kernel_refinement_df = pd.DataFrame(kernel_refinement_rows).set_index("kernel")
display(kernel_refinement_df)

fig, axis = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
kernel_error_comparison = kernel_refinement_df[[
    "coarse_mean_endpoint_error", "refined_mean_endpoint_error"
]].rename(columns={
    "coarse_mean_endpoint_error": f"{kernel_integration_steps} steps",
    "refined_mean_endpoint_error": f"{kernel_refinement_steps} steps",
})
kernel_error_comparison.plot.bar(ax=axis, rot=15, logy=True)
axis.set_ylabel("mean Euclidean endpoint error (log scale)")
axis.set_title("Guarded trajectory resolution check; inspect segment validity in the table")
axis.grid(axis="y", alpha=0.2)
plt.show()

The fields use a shared color scale and direction-normalized arrows. The two trajectory rows separate the raw field from a common annulus projection and boundary-cone guard. Projection can keep a poor field inside the annulus, so read endpoint errors together with the raw boundary diagnostic and unprojected paths. Unsupported queries are reported explicitly. The segment diagnostic also checks the closest point to the origin along each Euler segment: projecting endpoints alone can still allow a step across the hole. A penetration tolerance of $10^{-3}$ accounts for small boundary chords and floating-point error.

**Saved-run observations.** With 5,000 construction points, 12 shared starts, and $t=0.99$, the guarded 1,600-step mean endpoint errors are about 0.1287 (fixed Gaussian), 0.00131 (variable Gaussian), 0.1348 (polynomial), 0.1441 (bump), and 4.9333 (binary CkNN). The variable Gaussian has the smallest endpoint error, but its 800-step trajectories include a jump across the hole. At 1,600 steps the maximum hole penetration drops to about $6.4\times10^{-4}$, while its largest step is still 2.62. Its low endpoint error therefore does not establish convergence of the discontinuous-field ODE. The compact profiles have smaller maximum steps (about 0.169 and 0.274); their endpoint errors also change with refinement. Binary CkNN remains stationary up to projection roundoff.

Raw spectral distances have different scales across profiles; the decay plot therefore normalizes each path by its own initial distance. Euclidean endpoint error compares target convergence in the common annulus coordinates. These are deterministic toy comparisons on uniformly sampled construction points, not a density-robustness benchmark.

The fitted extensions, kernels, graphs, trajectories, and refinement results are available in kernel_models, kernel_fitted, kernel_graphs, kernel_trajectories, and kernel_refinement_df, indexed by the profile labels used in the plots.

## Train CellFlow RFM between the two Gaussian clouds

The left Gaussian is the source/control distribution and the right Gaussian is the target. The selected model from `kernel_models` supplies the same biharmonic geometry to OT matching and RFM interpolation:

$$c(x,y)=d_{\mathrm{BH}}(x,y)=\|\Psi(x)-\Psi(y)\|_2,\qquad \Psi_k(x)=\phi_k(x)/(\lambda_k+10^{-8}).$$

`match_biharmonic` evaluates the continuous Nyström embedding and calls the existing `cellflow.utils.match_linear` with the **unsquared** embedding distance. The standard `CellFlow(..., solver="otrfm") → prepare_data → prepare_model → train → predict` workflow keeps all cell coordinates two-dimensional and samples pairs from the OT coupling on each training batch. This is a linear OT problem with a manifold-aware cost.

Run the data-generation, construction, annulus-helper, and kernel-fit cells above first. The kernel comparison plots are optional. The compact network and training settings below suit this toy example. OT uses mean cost scaling, so `rfm_ot_epsilon` is relative to the batch's mean biharmonic cost. Training interpolation projects to the annulus; standard model prediction is unconstrained, and the evaluation reports its domain violations. Evaluation uses the fitted clouds and measures reconstruction, not generalization.

In [ ]:
import anndata as ad
import optax
from ott.geometry import costs
from scipy.spatial.distance import cdist

from cellflow.model import CellFlow
from cellflow.utils import match_linear

rfm_kernel_name = "fixed Gaussian"
rfm_nystroem = kernel_models[rfm_kernel_name]
rfm_ot_epsilon = 0.05
rfm_batch_size = 32
rfm_interpolation_steps = 300
rfm_num_iterations = 300
rfm_seed = 2026

rfm_source = left_gaussian_points.astype(np.float32)
rfm_target = right_gaussian_points.astype(np.float32)
assert kernel_spectral_weight_type == "biharmonic"
assert rfm_source.shape[1] == rfm_target.shape[1] == 2
assert np.all(np.isfinite(rfm_source)) and np.all(np.isfinite(rfm_target))
assert np.all(in_annulus(rfm_source)) and np.all(in_annulus(rfm_target))

# Embed each batch once, then let OTT form its pairwise cost matrix.
# Euclidean distance in these weighted coordinates IS biharmonic distance.
rfm_embed = jax.jit(jax.vmap(rfm_nystroem.extend_point))


def match_biharmonic(source_batch, target_batch):
    return match_linear(
        rfm_embed(jnp.asarray(source_batch)),
        rfm_embed(jnp.asarray(target_batch)),
        cost_fn=costs.Euclidean(),
        epsilon=rfm_ot_epsilon,
        scale_cost="mean",
        tau_a=1.0,
        tau_b=1.0,
        threshold=1e-5,
        max_iterations=5000,
    )


rfm_match_jit = jax.jit(match_biharmonic)
print(f"Geometry: {rfm_kernel_name}, {rfm_nystroem.Psi.shape[1]} biharmonic modes")
print(f"Source: {rfm_source.shape}; target: {rfm_target.shape}")
print(f"Batch size: {rfm_batch_size}; interpolation steps: {rfm_interpolation_steps}")
print(f"OT cost: unsquared biharmonic distance; relative epsilon: {rfm_ot_epsilon}")

In [ ]:
# Full-cloud diagnostic; training recomputes the coupling for each sampled batch.
rfm_source_embedding = np.asarray(rfm_embed(jnp.asarray(rfm_source)))
rfm_target_embedding = np.asarray(rfm_embed(jnp.asarray(rfm_target)))
rfm_cost_matrix = cdist(rfm_source_embedding, rfm_target_embedding)
rfm_coupling = np.asarray(rfm_match_jit(rfm_source, rfm_target))

assert rfm_coupling.shape == (len(rfm_source), len(rfm_target))
assert np.all(np.isfinite(rfm_coupling)) and np.all(rfm_coupling >= 0)
np.testing.assert_allclose(rfm_coupling.sum(axis=1), 1 / len(rfm_source), atol=5e-5)
np.testing.assert_allclose(rfm_coupling.sum(axis=0), 1 / len(rfm_target), atol=5e-5)

# Check that the OT cost agrees with the distance used by RFM.
rfm_check_distances = np.asarray(jax.jit(
    rfm_nystroem.distance_grad_and_norm_batch
)(jnp.asarray(rfm_source[:8]), jnp.asarray(rfm_target[:8]))[0])
np.testing.assert_allclose(
    np.diag(rfm_cost_matrix[:8, :8]), rfm_check_distances, rtol=1e-5, atol=1e-6
)
# A reordered, rectangular batch also works without global point indices.
rfm_rectangular_coupling = np.asarray(
    rfm_match_jit(rfm_source[7:0:-1], rfm_target[2:11])
)
assert rfm_rectangular_coupling.shape == (7, 9)
assert np.all(np.isfinite(rfm_rectangular_coupling))

display(pd.DataFrame({
    "coupling_mass": rfm_coupling.sum(),
    "max_source_marginal_error": np.max(np.abs(rfm_coupling.sum(axis=1) - 1 / len(rfm_source))),
    "max_target_marginal_error": np.max(np.abs(rfm_coupling.sum(axis=0) - 1 / len(rfm_target))),
    "OT_mean_biharmonic_cost": np.sum(rfm_coupling * rfm_cost_matrix),
    "independent_mean_biharmonic_cost": rfm_cost_matrix.mean(),
}, index=["coupling diagnostics"]).T)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, values, title in [
    (axes[0], rfm_cost_matrix, "Biharmonic cost"),
    (axes[1], rfm_coupling, "Entropic OT coupling"),
]:
    image = ax.imshow(values, aspect="auto", origin="lower")
    ax.set(xlabel="Target index", ylabel="Source index", title=title)
    fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# AnnData contains only the two Gaussian clouds; the uniform annulus is geometry.
rfm_n_source = len(rfm_source)
rfm_adata = ad.AnnData(
    X=np.vstack((rfm_source, rfm_target)),
    obs=pd.DataFrame(
        {
            "control": np.r_[np.ones(rfm_n_source, dtype=bool), np.zeros(len(rfm_target), dtype=bool)],
            "right_gaussian": np.r_[np.zeros(rfm_n_source, dtype=bool), np.ones(len(rfm_target), dtype=bool)],
        },
        index=[f"toy_{i}" for i in range(rfm_n_source + len(rfm_target))],
    ),
)

rfm_cf = CellFlow(rfm_adata, solver="otrfm")
rfm_cf.prepare_data(
    sample_rep="X",
    control_key="control",
    perturbation_covariates={"transport": ["right_gaussian"]},
)
rfm_cf.prepare_model(
    match_fn=match_biharmonic,
    nystroem=rfm_nystroem,
    condition_mode="deterministic",
    pooling="mean",
    layers_before_pool=[],
    layers_after_pool=[],
    condition_embedding_dim=16,
    cond_output_dropout=0.0,
    time_freqs=16,
    time_encoder_dims=(32, 32),
    hidden_dims=(64, 64),
    decoder_dims=(64, 64),
    conditioning_kwargs={},
    optimizer=optax.adam(1e-3),
    solver_kwargs={
        "interpolation_steps": rfm_interpolation_steps,
        "projection_fn": project_to_annulus,
        "project_to_manifold": False,
        "max_velocity_norm": 30.0,
    },
    seed=rfm_seed,
)

# The default dataloader must see left as source and right as target.
from cellflow.data._dataloader import TrainSampler

rfm_probe_batch = TrainSampler(rfm_cf.train_data, batch_size=rfm_batch_size).sample(
    np.random.default_rng(rfm_seed)
)
assert np.all(np.asarray(rfm_probe_batch["src_cell_data"])[:, 0] < 0)
assert np.all(np.asarray(rfm_probe_batch["tgt_cell_data"])[:, 0] > 0)
print("Solver:", type(rfm_cf.solver).__name__)
print("Coordinate dimensions:", rfm_cf.train_data.cell_data.shape[1])
print("Conditions:", rfm_cf.train_data.condition_data)

In [ ]:
# Baseline: leave the source cloud unchanged.
# These empirical distribution metrics do not assume pointwise correspondences.
def rfm_distribution_metrics(samples):
    samples = np.asarray(samples)
    energy = (
        2 * cdist(samples, rfm_target).mean()
        - cdist(samples, samples).mean()
        - cdist(rfm_target, rfm_target).mean()
    )
    return {
        "energy_distance_squared": float(max(energy, 0.0)),
        "mean_error_to_target": float(np.linalg.norm(samples.mean(axis=0) - rfm_target.mean(axis=0))),
        "outside_annulus_fraction": float(np.mean(~in_annulus(samples, inner_radius - 1e-4, outer_radius + 1e-4))),
    }


rfm_baseline_metrics = rfm_distribution_metrics(rfm_source)
print("Unchanged-source baseline:", rfm_baseline_metrics, flush=True)
rfm_training_started = time.perf_counter()
rfm_cf.train(
    num_iterations=rfm_num_iterations,
    batch_size=rfm_batch_size,
    valid_freq=100,
)
rfm_training_seconds = time.perf_counter() - rfm_training_started
rfm_losses = np.asarray(rfm_cf.trainer.training_logs["loss"])
assert len(rfm_losses) == rfm_num_iterations and np.all(np.isfinite(rfm_losses))
assert rfm_cf.solver.is_trained
print(f"Training finished in {rfm_training_seconds:.1f}s", flush=True)
print(f"Mean loss, first 25: {rfm_losses[:25].mean():.5f}; last 25: {rfm_losses[-25:].mean():.5f}")

In [ ]:
# Use the public prediction API on the left/control cloud.
rfm_source_adata = rfm_adata[rfm_adata.obs["control"]].copy()
rfm_predictions = rfm_cf.predict(
    rfm_source_adata,
    covariate_data=pd.DataFrame({"control": [True], "right_gaussian": [True], "condition": ["right Gaussian"]}),
    condition_id_key="condition",
)
rfm_predicted = np.asarray(rfm_predictions["right Gaussian"])
assert rfm_predicted.shape == rfm_source.shape
assert np.all(np.isfinite(rfm_predicted))
rfm_evaluation = pd.DataFrame(
    [rfm_baseline_metrics, rfm_distribution_metrics(rfm_predicted)],
    index=["unchanged source", "trained RFM prediction"],
)
display(rfm_evaluation)
print("Predictions use the standard unconstrained ODE solver; no endpoint projection is applied.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(rfm_losses, alpha=0.35, label="Training loss")
rfm_smooth_loss = pd.Series(rfm_losses).rolling(25, min_periods=1).mean()
axes[0].plot(rfm_smooth_loss, label="25-step mean")
axes[0].set(xlabel="Iteration", ylabel="Flow matching loss", title="RFM training")
axes[0].legend()
for ax, cloud, title, color in [
    (axes[1], rfm_source, "Source and target", "tab:blue"),
    (axes[2], rfm_predicted, "RFM prediction and target", "tab:green"),
]:
    ax.scatter(rfm_target[:, 0], rfm_target[:, 1], s=14, alpha=0.4, color="tab:orange", label="Right Gaussian target")
    ax.scatter(cloud[:, 0], cloud[:, 1], s=14, alpha=0.6, color=color, label="Source" if ax is axes[1] else "Prediction")
    ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, color="black"))
    ax.set(aspect="equal", xlabel="x", ylabel="y", title=title)
    ax.autoscale_view()
    ax.set_xlim(min(-3.3, cloud[:, 0].min() - 0.2), max(3.3, cloud[:, 0].max() + 0.2))
    ax.set_ylim(min(-3.3, cloud[:, 1].min() - 0.2), max(3.3, cloud[:, 1].max() + 0.2))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Reading the training result

The coupling checks verify balanced marginals and agreement between the OT cost and the RFM biharmonic distance. The prediction table compares empirical distributions without assuming a pointwise pairing between the two clouds.

In the initial 300-iteration run above, squared energy distance decreased from 8.54 to 0.49 and mean-position error from 4.93 to 0.58. However, 64.8% of predicted endpoints lay inside the annulus hole. The public `predict` method integrates the learned field with its standard unconstrained ODE solver; `projection_fn` in `solver_kwargs` only projects the training interpolations. Further training or a constrained prediction integrator is needed to obtain an annulus-preserving learned transport. The displayed metrics and plots should be checked again whenever the kernel, regularization, interpolation resolution, or training length changes.

## Inspect the learned vector field at a point and time

Set `rfm_query_point` to the current position $x_t$ and `rfm_field_time` to a time in $[0,1]$, then rerun the cell below. It evaluates $v_	heta(t,x_t\mid\mathrm{right\ Gaussian})$ using the trained inference weights, the same condition as prediction, and dropout disabled.

The colored arrows show the learned field on an annulus grid; the red arrow highlights the selected point. Colors encode speed. All arrow lengths equal `arrow_dt` times velocity, so their magnitudes remain comparable. These are instantaneous velocities. The plot uses the field directly, including any components pointing across a boundary.

In [ ]:
# Change these two values and rerun this cell.
rfm_query_point = np.array([0, -2.5], dtype=np.float32)
rfm_field_time = 0.5


def plot_rfm_learned_field(point, time, grid_size=27, arrow_dt=0.04):
    """Plot the trained instantaneous velocity and highlight v(time, point)."""
    if not rfm_cf.solver.is_trained:
        raise ValueError("Run the RFM training cell first.")
    point = np.asarray(point, dtype=np.float32)
    if point.shape != (2,) or not np.all(np.isfinite(point)):
        raise ValueError("point must contain two finite coordinates.")
    if not np.isfinite(time) or not 0.0 <= time <= 1.0:
        raise ValueError("time must be between 0 and 1.")
    if grid_size < 3 or arrow_dt <= 0:
        raise ValueError("grid_size must be at least 3 and arrow_dt must be positive.")

    axis_values = np.linspace(-outer_radius, outer_radius, grid_size, dtype=np.float32)
    grid_x, grid_y = np.meshgrid(axis_values, axis_values)
    grid = np.column_stack((grid_x.ravel(), grid_y.ravel()))
    grid = grid[in_annulus(grid, inner_radius, outer_radius)]
    query_points = jnp.asarray(np.vstack((grid, point[None])))

    # This toy model has one target condition. Use the same shape and inference
    # state as OTRFlowMatching.predict, with deterministic condition-encoder noise.
    condition = {
        key: jnp.asarray(values[:1])
        for key, values in rfm_cf.train_data.condition_data.items()
    }
    state = rfm_cf.solver.vf_state_inference
    encoder_noise = jnp.zeros((1, rfm_cf.solver.vf.condition_embedding_dim))

    @jax.jit
    def evaluate_field(positions, query_time):
        times = jnp.full((positions.shape[0], 1), query_time, dtype=positions.dtype)
        return state.apply_fn(
            {"params": state.params},
            times,
            positions,
            condition,
            encoder_noise=encoder_noise,
            train=False,
        )[0]

    velocities = np.asarray(evaluate_field(query_points, jnp.asarray(time)))
    if not np.all(np.isfinite(velocities)):
        raise ValueError("The learned field contains non-finite velocities.")
    grid_velocities, point_velocity = velocities[:-1], velocities[-1]
    speeds = np.linalg.norm(grid_velocities, axis=1)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(rfm_source[:, 0], rfm_source[:, 1], s=6, alpha=0.15, color="tab:blue", label="Source cloud (t=0)")
    ax.scatter(rfm_target[:, 0], rfm_target[:, 1], s=6, alpha=0.15, color="tab:orange", label="Target cloud (t=1)")
    arrows = ax.quiver(
        grid[:, 0], grid[:, 1], grid_velocities[:, 0], grid_velocities[:, 1], speeds,
        cmap="viridis", angles="xy", scale_units="xy", scale=1 / arrow_dt,
        width=0.003, alpha=0.85,
    )
    fig.colorbar(arrows, ax=ax, label="Learned speed ||v(t, x)||")
    ax.scatter(*point, s=70, color="tab:red", edgecolor="white", zorder=5, label="Selected point")
    ax.quiver(
        point[0], point[1], point_velocity[0], point_velocity[1],
        angles="xy", scale_units="xy", scale=1 / arrow_dt,
        color="tab:red", width=0.008, zorder=6,
    )
    ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, edgecolor="gray"))
    arrow_tip = point + arrow_dt * point_velocity
    ax.set(
        aspect="equal", xlabel="x", ylabel="y",
        title=(f"Learned RFM field at t={time:.2f}" + chr(10)
               + f"x=({point[0]:.2f}, {point[1]:.2f}), v=({point_velocity[0]:.3f}, {point_velocity[1]:.3f})"),
        xlim=(min(-3.4, point[0] - 0.4, arrow_tip[0] - 0.4), max(3.4, point[0] + 0.4, arrow_tip[0] + 0.4)),
        ylim=(min(-3.4, point[1] - 0.4, arrow_tip[1] - 0.4), max(3.4, point[1] + 0.4, arrow_tip[1] + 0.4)),
    )
    ax.text(0.02, 0.02, f"Arrow length = {arrow_dt:g} × velocity", transform=ax.transAxes, fontsize=9)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()
    return fig, ax, point_velocity


rfm_field_fig, rfm_field_ax, rfm_query_velocity = plot_rfm_learned_field(
    rfm_query_point, rfm_field_time
)
print(f"v(t={rfm_field_time}, x={rfm_query_point.tolist()}) = {rfm_query_velocity}")
print(f"Speed: {np.linalg.norm(rfm_query_velocity):.6f}")

## Learned trajectories from new source samples

Draw fresh points from the same Gaussian as the source dataset, conditioned to lie in the annulus. Change `rfm_n_new_points` or `rfm_trajectory_seed` below to choose a different sample. Each trajectory solves $\dot{x}_t=v_	heta(t,x_t\mid\mathrm{right\ Gaussian})$ from $t=0$ to $t=1$ using the trained inference weights and the standard predictor's Tsit5 solver and tolerances. Saving intermediate states makes the full paths visible.

The circles mark starting points, crosses mark endpoints, and small arrows indicate travel direction. The radius plot shows when a path leaves the annulus. The trajectories use the learned field directly, so boundary crossings are retained.

In [ ]:
import diffrax

rfm_n_new_points = 6
rfm_trajectory_seed = 2027
rfm_trajectory_times = np.linspace(0.0, 1.0, 201, dtype=np.float32)

if not rfm_cf.solver.is_trained:
    raise ValueError("Run the RFM training cell first.")
if rfm_n_new_points < 1:
    raise ValueError("rfm_n_new_points must be positive.")

# Fresh draws from the source distribution, independent of the training cloud.
rfm_new_start_points, _, _ = sample_gaussian_on_annulus(
    mean_left, variance, rfm_n_new_points,
    np.random.default_rng(rfm_trajectory_seed),
    inner_radius=inner_radius, outer_radius=outer_radius,
)
rfm_new_start_points = rfm_new_start_points.astype(np.float32)
assert np.all(in_annulus(rfm_new_start_points, inner_radius, outer_radius))
assert not np.any(np.all(
    rfm_new_start_points[:, None, :] == rfm_source[None, :, :], axis=2
))

rfm_path_condition = {
    key: jnp.asarray(values[:1])
    for key, values in rfm_cf.train_data.condition_data.items()
}
rfm_path_state = rfm_cf.solver.vf_state_inference
rfm_path_encoder_noise = jnp.zeros((1, rfm_cf.solver.vf.condition_embedding_dim))


def rfm_learned_path_velocity(t, position, args):
    params, condition, encoder_noise = args
    return rfm_path_state.apply_fn(
        {"params": params},
        t,
        position,
        condition,
        encoder_noise,
        train=False,
    )[0]


def rfm_integrate_new_point(params, start, condition, encoder_noise):
    solution = diffrax.diffeqsolve(
        diffrax.ODETerm(rfm_learned_path_velocity),
        solver=diffrax.Tsit5(),
        t0=0.0,
        t1=1.0,
        dt0=None,
        y0=start,
        args=(params, condition, encoder_noise),
        stepsize_controller=diffrax.PIDController(rtol=1e-5, atol=1e-5),
        saveat=diffrax.SaveAt(ts=jnp.asarray(rfm_trajectory_times)),
    )
    return solution.ys, solution.stats["num_steps"]


# Shape: (number of new points, number of saved times, 2).
rfm_new_trajectories, rfm_new_solver_steps = jax.jit(
    jax.vmap(rfm_integrate_new_point, in_axes=(None, 0, None, None))
)(rfm_path_state.params, jnp.asarray(rfm_new_start_points),
  rfm_path_condition, rfm_path_encoder_noise)
rfm_new_trajectories = np.asarray(rfm_new_trajectories)
assert rfm_new_trajectories.shape == (rfm_n_new_points, len(rfm_trajectory_times), 2)
assert np.all(np.isfinite(rfm_new_trajectories))
np.testing.assert_allclose(rfm_new_trajectories[:, 0], rfm_new_start_points)

# Check that saving the paths gives the same endpoints as standard prediction.
rfm_new_predicted_endpoints = rfm_cf.solver.predict(
    rfm_new_start_points, rfm_path_condition
)
np.testing.assert_allclose(
    rfm_new_trajectories[:, -1], rfm_new_predicted_endpoints, rtol=1e-4, atol=1e-4
)
print(f"Integrated {rfm_n_new_points} new source samples, with {len(rfm_trajectory_times)} saved times each.")
print("Maximum endpoint difference from standard prediction:",
      np.max(np.abs(rfm_new_trajectories[:, -1] - rfm_new_predicted_endpoints)))

In [ ]:
from matplotlib.lines import Line2D

rfm_new_path_radii = np.linalg.norm(rfm_new_trajectories, axis=2)
rfm_new_path_in_annulus = (
    (rfm_new_path_radii >= inner_radius - 1e-4)
    & (rfm_new_path_radii <= outer_radius + 1e-4)
)
rfm_new_path_summary = pd.DataFrame(
    {
        "start_x": rfm_new_start_points[:, 0],
        "start_y": rfm_new_start_points[:, 1],
        "end_x": rfm_new_trajectories[:, -1, 0],
        "end_y": rfm_new_trajectories[:, -1, 1],
        "min_sampled_radius": rfm_new_path_radii.min(axis=1),
        "max_sampled_radius": rfm_new_path_radii.max(axis=1),
        "endpoint_in_annulus": rfm_new_path_in_annulus[:, -1],
        "all_saved_points_in_annulus": rfm_new_path_in_annulus.all(axis=1),
        "adaptive_solver_steps": np.asarray(rfm_new_solver_steps),
    },
    index=[f"Path {i + 1}" for i in range(rfm_n_new_points)],
)
display(rfm_new_path_summary)

fig, (path_ax, radius_ax) = plt.subplots(1, 2, figsize=(13, 6))
path_ax.scatter(rfm_source[:, 0], rfm_source[:, 1], s=9, alpha=0.2,
                color="tab:blue", label="Source cloud")
path_ax.scatter(rfm_target[:, 0], rfm_target[:, 1], s=9, alpha=0.2,
                color="tab:orange", label="Target cloud")
path_ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False,
                            linestyle="--", color="tab:red"))
path_ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, edgecolor="gray"))
radius_ax.axhspan(inner_radius, outer_radius, color="tab:green", alpha=0.1,
                 label="Annulus")
radius_ax.axhline(inner_radius, color="tab:red", linestyle="--")
radius_ax.axhline(outer_radius, color="gray", linestyle="--")

rfm_path_colors = plt.get_cmap("tab10")
for path_idx, path in enumerate(rfm_new_trajectories):
    color = rfm_path_colors(path_idx % 10)
    label = f"Path {path_idx + 1}"
    path_ax.plot(path[:, 0], path[:, 1], color=color, linewidth=2, label=label)
    path_ax.scatter(*path[0], s=65, color=color, edgecolor="white", zorder=5)
    path_ax.scatter(*path[-1], marker="X", s=85, color=color, edgecolor="white", zorder=6)
    # Arrows follow the curve at evenly spaced saved times.
    for time_idx in np.linspace(25, len(path) - 15, 4, dtype=int):
        path_ax.annotate(
            "", xy=path[time_idx + 4], xytext=path[time_idx],
            arrowprops={"arrowstyle": "->", "color": color, "lw": 1.8},
        )
    radius_ax.plot(rfm_trajectory_times, rfm_new_path_radii[path_idx],
                   color=color, linewidth=2, label=label)

rfm_path_extent = max(3.3, float(np.max(np.abs(rfm_new_trajectories))) + 0.2)
path_ax.set(
    aspect="equal", xlabel="x", ylabel="y",
    xlim=(-rfm_path_extent, rfm_path_extent),
    ylim=(-rfm_path_extent, rfm_path_extent),
    title="Learned RFM trajectories from fresh source samples",
)
handles, labels = path_ax.get_legend_handles_labels()
handles.extend([
    Line2D([], [], linestyle="none", marker="o", color="gray", label="Start (t=0)"),
    Line2D([], [], linestyle="none", marker="X", color="gray", label="End (t=1)"),
])
path_ax.legend(handles=handles, loc="upper right", fontsize=8)
radius_ax.set(xlabel="Time t", ylabel="Radius ||x(t)||", xlim=(0, 1),
              title="Distance from the origin along each path")
radius_ax.legend(fontsize=8, loc="best")
for ax in (path_ax, radius_ax):
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
print(f"{np.count_nonzero(~rfm_new_path_in_annulus.all(axis=1))}/{rfm_n_new_points} paths leave the annulus at a saved time.")

## Four-way comparison: Riemannian vs. Euclidean Flow Matching, biharmonic vs. Euclidean couplings

The RFM model above combines two geometry-aware ingredients: a **biharmonic OT coupling**, which decides which source and target points are paired, and **Riemannian interpolation**, which decides how each pair is connected. This section separates the two by training all four combinations on the same source and target clouds:

| | Biharmonic coupling $c_{\mathrm{BH}}(x,y)=\|\Psi(x)-\Psi(y)\|_2$ | Euclidean coupling $c_{\mathrm{E}}(x,y)=\|x-y\|_2$ |
| --- | --- | --- |
| **Riemannian FM** (`solver="otrfm"`) | `rfm_cf` (trained above) | `rfm_euclidean_cf` |
| **Euclidean FM** (`solver="otfm"`) | `fm_biharmonic_cf` | `fm_euclidean_cf` |

Both RFM models use the same Nyström geometry and the same `RFMInterpolation` settings as above: annulus projection of the training interpolations, `interpolation_steps`, and velocity clipping. Both Euclidean models use the existing zero-noise constant-noise path
$$x_t=(1-t)x_0+tx_1,\qquad u_t=x_1-x_0,$$
whose straight segments ignore the annulus. For most left-to-right pairs they cross the hole.

Everything else is shared. The network architecture and optimizer transformation are copied from `rfm_cf`. The time sampler, EMA, batch size, iteration count, and initialization seed also match. Both couplings are balanced and entropic, use unsquared costs, and share the relative `rfm_ot_epsilon` with mean cost scaling. Each coupling is recomputed on every training batch.

Run the RFM training, prediction, query-field, and fresh-sample trajectory cells above first. The extra RFM model costs about as much training time as the first because each step integrates the RFM interpolation. The Euclidean models train much faster. Training losses regress different velocity targets, so they are not comparable across interpolation types. Compare the endpoint distribution metrics, annulus violations, and path geometry instead.

In [ ]:
# Two entropic OT couplings with identical settings; only the ground cost differs.
cmp_ot_epsilon = rfm_ot_epsilon
cmp_batch_size = rfm_batch_size
cmp_num_iterations = rfm_num_iterations
cmp_seed = rfm_seed


def match_euclidean(source_batch, target_batch):
    return match_linear(
        jnp.asarray(source_batch),
        jnp.asarray(target_batch),
        cost_fn=costs.Euclidean(),
        epsilon=cmp_ot_epsilon,
        scale_cost="mean",
        tau_a=1.0,
        tau_b=1.0,
        threshold=1e-5,
        max_iterations=5000,
    )


cmp_match_euclidean_jit = jax.jit(match_euclidean)
cmp_cost_matrices = {
    "biharmonic": rfm_cost_matrix,
    "Euclidean": cdist(rfm_source, rfm_target),
}
cmp_couplings = {
    "biharmonic": rfm_coupling,
    "Euclidean": np.asarray(cmp_match_euclidean_jit(rfm_source, rfm_target)),
}
np.testing.assert_allclose(
    np.diag(cmp_cost_matrices["Euclidean"][:8, :8]),
    np.linalg.norm(rfm_source[:8] - rfm_target[:8], axis=1),
    rtol=1e-6, atol=1e-6,
)
# A reordered, rectangular batch also works without global point indices.
cmp_rectangular_coupling = np.asarray(
    cmp_match_euclidean_jit(rfm_source[7:0:-1], rfm_target[2:11])
)
assert cmp_rectangular_coupling.shape == (7, 9)
assert np.all(np.isfinite(cmp_rectangular_coupling))


def straight_segment_min_radius(starts, ends):
    """Smallest distance to the origin along each straight segment (broadcasts)."""
    segments = ends - starts
    fractions = np.clip(
        -np.sum(starts * segments, axis=-1)
        / np.maximum(np.sum(segments**2, axis=-1), 1e-20),
        0.0, 1.0,
    )
    return np.linalg.norm(starts + fractions[..., None] * segments, axis=-1)


# Pair-level diagnostics: does the straight chord cross the hole, and do the
# two endpoints lie on the same side (upper/lower half) of the annulus?
cmp_pair_min_radius = straight_segment_min_radius(
    rfm_source[:, None, :], rfm_target[None, :, :]
)
cmp_same_half = np.sign(rfm_source[:, None, 1]) == np.sign(rfm_target[None, :, 1])
cmp_independent_coupling = np.full(
    (len(rfm_source), len(rfm_target)), 1.0 / (len(rfm_source) * len(rfm_target))
)
cmp_coupling_rows = []
for name, coupling in [*cmp_couplings.items(), ("independent", cmp_independent_coupling)]:
    assert coupling.shape == (len(rfm_source), len(rfm_target))
    assert np.all(np.isfinite(coupling)) and np.all(coupling >= 0.0)
    np.testing.assert_allclose(coupling.sum(axis=1), 1 / len(rfm_source), atol=5e-5)
    np.testing.assert_allclose(coupling.sum(axis=0), 1 / len(rfm_target), atol=5e-5)
    mass = coupling / coupling.sum()
    cmp_coupling_rows.append({
        "coupling": name,
        "coupling_mass": coupling.sum(),
        "max_source_marginal_error": np.max(np.abs(coupling.sum(axis=1) - 1 / len(rfm_source))),
        "max_target_marginal_error": np.max(np.abs(coupling.sum(axis=0) - 1 / len(rfm_target))),
        "mean_biharmonic_cost": np.sum(mass * cmp_cost_matrices["biharmonic"]),
        "mean_euclidean_cost": np.sum(mass * cmp_cost_matrices["Euclidean"]),
        "same_half_mass": np.sum(mass * cmp_same_half),
        "chord_through_hole_mass": np.sum(mass * (cmp_pair_min_radius < inner_radius)),
    })
cmp_coupling_df = pd.DataFrame(cmp_coupling_rows).set_index("coupling")
display(cmp_coupling_df)

# Sort both clouds by height so that pairing structure is visible in the matrices.
cmp_source_order = np.argsort(rfm_source[:, 1])
cmp_target_order = np.argsort(rfm_target[:, 1])
cmp_pair_rng = np.random.default_rng(cmp_seed)
cmp_pair_sources = cmp_pair_rng.choice(len(rfm_source), 40, replace=False)
fig, axes = plt.subplots(2, 2, figsize=(12, 11), constrained_layout=True)
for column, (name, coupling) in enumerate(cmp_couplings.items()):
    image = axes[0, column].imshow(
        coupling[np.ix_(cmp_source_order, cmp_target_order)],
        aspect="auto", origin="lower",
    )
    axes[0, column].set(
        xlabel="Target index (sorted by y)", ylabel="Source index (sorted by y)",
        title=f"{name} entropic OT coupling",
    )
    fig.colorbar(image, ax=axes[0, column])

    ax = axes[1, column]
    ax.scatter(*rfm_source.T, s=8, alpha=0.25, color="tab:blue", label="Source")
    ax.scatter(*rfm_target.T, s=8, alpha=0.25, color="tab:orange", label="Target")
    partners = coupling[cmp_pair_sources].argmax(axis=1)
    for source_index, target_index in zip(cmp_pair_sources, partners):
        ax.plot(
            [rfm_source[source_index, 0], rfm_target[target_index, 0]],
            [rfm_source[source_index, 1], rfm_target[target_index, 1]],
            color="gray", alpha=0.5, linewidth=0.8,
        )
    ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, color="black"))
    ax.set(
        aspect="equal", xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), xlabel="x", ylabel="y",
        title=f"{name}: most likely partner of 40 sources\n(lines show pairs, not paths)",
    )
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.2)
plt.show()

In [ ]:
from cellflow.solvers import OTFlowMatching
from cellflow.solvers._otrfm import OTRFlowMatching

# Copy the fitted RFM model's architecture, including its default settings.
# The optimizer transformation is shared; every new model initializes fresh
# parameters and optimizer state from the same initialization seed.
cmp_architecture_names = (
    "condition_mode", "regularization", "condition_embedding_dim",
    "pooling", "pooling_kwargs", "layers_before_pool", "layers_after_pool",
    "cond_output_dropout", "condition_encoder_kwargs",
    "time_freqs", "time_max_period", "time_encoder_dims", "time_encoder_dropout",
    "hidden_dims", "hidden_dropout", "conditioning", "conditioning_kwargs",
    "decoder_dims", "decoder_dropout",
    "layer_norm_before_concatenation", "linear_projection_before_concatenation",
)
cmp_architecture_kwargs = {
    name: getattr(rfm_cf.vf, name) for name in cmp_architecture_names
}
cmp_architecture_kwargs["vf_act_fn"] = rfm_cf.vf.act_fn
cmp_shared_solver_kwargs = {
    "time_sampler": rfm_cf.solver.time_sampler,
    "ema": rfm_cf.solver.ema,
}
# Identical Riemannian interpolation settings to the biharmonic RFM model.
cmp_rfm_solver_kwargs = {
    **cmp_shared_solver_kwargs,
    "interpolation_steps": rfm_cf.solver.interpolation_steps,
    "projection_fn": rfm_cf.solver.projection_fn,
    "project_to_manifold": rfm_cf.solver.project_to_manifold,
    "min_gradient_norm_sq": rfm_cf.solver.min_gradient_norm_sq,
    "time_epsilon": rfm_cf.solver.time_epsilon,
    "max_velocity_norm": rfm_cf.solver.max_velocity_norm,
}


def build_comparison_model(solver, match_fn):
    model = CellFlow(rfm_adata.copy(), solver=solver)
    model.prepare_data(
        sample_rep="X",
        control_key="control",
        perturbation_covariates={"transport": ["right_gaussian"]},
    )
    if solver == "otrfm":
        model.prepare_model(
            **cmp_architecture_kwargs,
            match_fn=match_fn,
            nystroem=rfm_nystroem,
            optimizer=rfm_cf.solver.vf_state.tx,
            solver_kwargs=cmp_rfm_solver_kwargs,
            seed=cmp_seed,
        )
        assert type(model.solver) is OTRFlowMatching
        assert model.solver.nystroem is rfm_nystroem
    else:
        model.prepare_model(
            **cmp_architecture_kwargs,
            match_fn=match_fn,
            probability_path={"constant_noise": 0.0},
            optimizer=rfm_cf.solver.vf_state.tx,
            solver_kwargs=cmp_shared_solver_kwargs,
            seed=cmp_seed,
        )
        assert type(model.solver) is OTFlowMatching
        assert model.solver.probability_path.sigma == 0.0
    assert model.vf == rfm_cf.vf
    assert int(model.solver.vf_state.step) == 0
    np.testing.assert_array_equal(model.train_data.cell_data, rfm_cf.train_data.cell_data)
    for key in rfm_cf.train_data.condition_data:
        np.testing.assert_array_equal(
            model.train_data.condition_data[key], rfm_cf.train_data.condition_data[key]
        )
    # The default dataloader must see left as source and right as target.
    probe_batch = TrainSampler(model.train_data, batch_size=cmp_batch_size).sample(
        np.random.default_rng(cmp_seed)
    )
    assert np.all(np.asarray(probe_batch["src_cell_data"])[:, 0] < 0)
    assert np.all(np.asarray(probe_batch["tgt_cell_data"])[:, 0] > 0)
    return model


rfm_euclidean_cf = build_comparison_model("otrfm", match_euclidean)
fm_biharmonic_cf = build_comparison_model("otfm", match_biharmonic)
fm_euclidean_cf = build_comparison_model("otfm", match_euclidean)

cmp_models = {
    "RFM + biharmonic OT": rfm_cf,
    "RFM + Euclidean OT": rfm_euclidean_cf,
    "FM + biharmonic OT": fm_biharmonic_cf,
    "FM + Euclidean OT": fm_euclidean_cf,
}
cmp_colors = {
    "RFM + biharmonic OT": "tab:green",
    "RFM + Euclidean OT": "tab:cyan",
    "FM + biharmonic OT": "tab:purple",
    "FM + Euclidean OT": "tab:pink",
}
for name, model in cmp_models.items():
    print(f"{name}: {type(model.solver).__name__}")
print(f"Iterations: {cmp_num_iterations}; batch size: {cmp_batch_size}; seed: {cmp_seed}")
print(f"Unsquared OT costs; relative epsilon: {cmp_ot_epsilon}")

In [ ]:
# The biharmonic RFM model is already trained; train the other three.
cmp_losses = {"RFM + biharmonic OT": rfm_losses}
cmp_training_seconds = {"RFM + biharmonic OT": rfm_training_seconds}
for name, model in cmp_models.items():
    if name in cmp_losses:
        continue
    print(f"Training {name}", flush=True)
    training_started = time.perf_counter()
    model.train(
        num_iterations=cmp_num_iterations,
        batch_size=cmp_batch_size,
        valid_freq=100,
    )
    cmp_training_seconds[name] = time.perf_counter() - training_started
    losses = np.asarray(model.trainer.training_logs["loss"])
    assert len(losses) == cmp_num_iterations and np.all(np.isfinite(losses))
    assert model.solver.is_trained
    assert int(model.solver.vf_state.step) == cmp_num_iterations
    cmp_losses[name] = losses
    print(
        f"{name}: {cmp_training_seconds[name]:.1f}s; mean loss first 25 "
        f"{losses[:25].mean():.5f}, last 25 {losses[-25:].mean():.5f}",
        flush=True,
    )

display(pd.DataFrame({
    "training_seconds": cmp_training_seconds,
    "mean_loss_first_25": {name: losses[:25].mean() for name, losses in cmp_losses.items()},
    "mean_loss_last_25": {name: losses[-25:].mean() for name, losses in cmp_losses.items()},
}).loc[list(cmp_models)])

In [ ]:
# The public prediction API and target condition match the RFM evaluation.
cmp_source_adata = rfm_adata[rfm_adata.obs["control"]].copy()
cmp_covariate_data = pd.DataFrame({
    "control": [True], "right_gaussian": [True], "condition": ["right Gaussian"],
})
cmp_predicted = {}
for name, model in cmp_models.items():
    predictions = model.predict(
        cmp_source_adata, covariate_data=cmp_covariate_data, condition_id_key="condition",
    )
    cmp_predicted[name] = np.asarray(predictions["right Gaussian"])
    assert cmp_predicted[name].shape == rfm_source.shape
    assert np.all(np.isfinite(cmp_predicted[name]))
np.testing.assert_allclose(
    cmp_predicted["RFM + biharmonic OT"], rfm_predicted, rtol=1e-4, atol=1e-4
)


def annulus_violation(samples):
    radii = np.linalg.norm(samples, axis=-1)
    return np.maximum(inner_radius - radii, 0.0) + np.maximum(radii - outer_radius, 0.0)


def comparison_distribution_metrics(samples):
    radii = np.linalg.norm(samples, axis=-1)
    violation = annulus_violation(samples)
    return {
        **rfm_distribution_metrics(samples),
        "inside_hole_fraction": float(np.mean(radii < inner_radius - 1e-4)),
        "beyond_outer_fraction": float(np.mean(radii > outer_radius + 1e-4)),
        "mean_annulus_violation": float(violation.mean()),
        "p95_annulus_violation": float(np.quantile(violation, 0.95)),
    }


cmp_endpoint_metrics = pd.DataFrame(
    [comparison_distribution_metrics(rfm_source)]
    + [comparison_distribution_metrics(samples) for samples in cmp_predicted.values()],
    index=["unchanged source", *cmp_predicted],
)
display(cmp_endpoint_metrics)

fig, axes = plt.subplots(1, 4, figsize=(20, 4), constrained_layout=True)
for ax, (name, losses) in zip(axes, cmp_losses.items()):
    ax.plot(losses, alpha=0.3, color=cmp_colors[name], label="Training loss")
    ax.plot(pd.Series(losses).rolling(25, min_periods=1).mean(), color="black",
            linewidth=1.2, label="25-step mean")
    ax.set(xlabel="Iteration", ylabel="Flow matching loss", title=name)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.suptitle("Training losses (different velocity targets; compare shapes, not values across rows)")
plt.show()

cmp_extent = max(3.3, float(np.max(np.abs(np.vstack(list(cmp_predicted.values()))))) + 0.2)
fig, axes = plt.subplots(1, 5, figsize=(25, 5.5), constrained_layout=True)
for ax, (title, samples, color) in zip(axes, [
    ("Source and target", rfm_source, "tab:blue"),
    *[(name, samples, cmp_colors[name]) for name, samples in cmp_predicted.items()],
]):
    ax.scatter(*rfm_target.T, s=12, alpha=0.35, color="tab:orange", label="Right Gaussian target")
    ax.scatter(*samples.T, s=12, alpha=0.6, color=color,
               label="Source" if title == "Source and target" else "Prediction")
    ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, color="black"))
    if title in cmp_endpoint_metrics.index and title != "unchanged source":
        metrics = cmp_endpoint_metrics.loc[title]
        title = (f"{title}\nenergy² {metrics['energy_distance_squared']:.3f}, "
                 f"outside {metrics['outside_annulus_fraction']:.0%}")
    ax.set(aspect="equal", xlim=(-cmp_extent, cmp_extent), ylim=(-cmp_extent, cmp_extent),
           xlabel="x", ylabel="y", title=title)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.2)
fig.suptitle("Same data, architecture, optimizer, batch size, seed, and iteration count")
plt.show()
print("All predictions use standard unconstrained ODE integration; no endpoint projection is applied.")

### Compare the learned fields and fresh-sample paths

All four learned fields below use the same query point and time as the single-model field plot above (`rfm_query_point`, `rfm_field_time`). They also share the annulus grid, arrow scale, and speed color scale. Arrow length is `cmp_arrow_dt` times velocity, so magnitudes are comparable across panels.

The path comparison reuses `rfm_new_start_points` and `rfm_trajectory_times` exactly. Every model integrates its own learned field with the same Tsit5 solver and tolerances as the public `predict` method, with no annulus projection. The biharmonic RFM paths are recomputed and checked against `rfm_new_trajectories`. Segment-level hole penetration uses a closest-point test on each straight segment between saved times, so it also detects chords that cut through the hole between two saved times.

In [ ]:
cmp_query_point = np.asarray(rfm_query_point, dtype=np.float32)
cmp_field_time = rfm_field_time
cmp_arrow_dt = 0.04
cmp_axis_values = np.linspace(-outer_radius, outer_radius, 27, dtype=np.float32)
cmp_grid_x, cmp_grid_y = np.meshgrid(cmp_axis_values, cmp_axis_values)
cmp_field_grid = np.column_stack((cmp_grid_x.ravel(), cmp_grid_y.ravel()))
cmp_field_grid = cmp_field_grid[in_annulus(cmp_field_grid, inner_radius, outer_radius)]
cmp_field_queries = jnp.asarray(np.vstack((cmp_field_grid, cmp_query_point[None])))


def evaluate_comparison_field(model, positions, query_time):
    """Evaluate v(t, x | right Gaussian) with the inference weights and no dropout."""
    state = model.solver.vf_state_inference
    condition = {
        key: jnp.asarray(values[:1])
        for key, values in model.train_data.condition_data.items()
    }
    encoder_noise = jnp.zeros((1, model.solver.vf.condition_embedding_dim))

    @jax.jit
    def evaluate(query_positions):
        times = jnp.full((query_positions.shape[0], 1), query_time, dtype=query_positions.dtype)
        return state.apply_fn(
            {"params": state.params}, times, query_positions, condition,
            encoder_noise=encoder_noise, train=False,
        )[0]

    return np.asarray(evaluate(positions))


cmp_fields = {
    name: evaluate_comparison_field(model, cmp_field_queries, cmp_field_time)
    for name, model in cmp_models.items()
}
assert all(np.all(np.isfinite(values)) for values in cmp_fields.values())
cmp_speed_max = max(
    float(np.linalg.norm(values[:-1], axis=1).max()) for values in cmp_fields.values()
)
fig, axes = plt.subplots(2, 2, figsize=(15, 13), constrained_layout=True)
for ax, (name, velocities) in zip(axes.ravel(), cmp_fields.items()):
    grid_velocities, point_velocity = velocities[:-1], velocities[-1]
    arrows = ax.quiver(
        cmp_field_grid[:, 0], cmp_field_grid[:, 1],
        grid_velocities[:, 0], grid_velocities[:, 1],
        np.linalg.norm(grid_velocities, axis=1),
        cmap="viridis", clim=(0.0, max(cmp_speed_max, 1e-6)),
        angles="xy", scale_units="xy", scale=1 / cmp_arrow_dt,
        width=0.003, alpha=0.85,
    )
    ax.scatter(*rfm_source.T, s=6, alpha=0.15, color="tab:blue", label="Source cloud (t=0)")
    ax.scatter(*rfm_target.T, s=6, alpha=0.15, color="tab:orange", label="Target cloud (t=1)")
    ax.scatter(*cmp_query_point, s=70, color="tab:red", edgecolor="white", zorder=5,
               label="Query point")
    ax.quiver(
        *cmp_query_point, *point_velocity,
        angles="xy", scale_units="xy", scale=1 / cmp_arrow_dt,
        color="tab:red", width=0.008, zorder=6,
    )
    ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, edgecolor="gray"))
    ax.set(
        aspect="equal", xlabel="x", ylabel="y", xlim=(-3.4, 3.4), ylim=(-3.4, 3.4),
        title=(f"{name} at t={cmp_field_time:.2f}\n"
               f"v(query) = ({point_velocity[0]:.3f}, {point_velocity[1]:.3f})"),
    )
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.2)
fig.colorbar(arrows, ax=list(axes.ravel()), label="Learned speed ||v(t, x)||, shared scale",
             shrink=0.6)
fig.suptitle(f"Arrow length = {cmp_arrow_dt:g} × velocity; common point and time")
plt.show()
display(pd.DataFrame(
    [values[-1] for values in cmp_fields.values()],
    index=list(cmp_fields), columns=["query_vx", "query_vy"],
).assign(query_speed=lambda df: np.hypot(df["query_vx"], df["query_vy"])))

In [ ]:
# Reuse the same fresh source points and saved times as the RFM section above.
cmp_new_start_points = rfm_new_start_points.copy()
cmp_trajectory_times = rfm_trajectory_times.copy()


def integrate_comparison_paths(model, starts, times):
    """Integrate the learned field from each start, saving the given times."""
    state = model.solver.vf_state_inference
    condition = {
        key: jnp.asarray(values[:1])
        for key, values in model.train_data.condition_data.items()
    }
    encoder_noise = jnp.zeros((1, model.solver.vf.condition_embedding_dim))

    def velocity(t, position, params):
        return state.apply_fn(
            {"params": params}, t, position, condition, encoder_noise, train=False,
        )[0]

    def integrate_one(params, start):
        solution = diffrax.diffeqsolve(
            diffrax.ODETerm(velocity),
            solver=diffrax.Tsit5(),
            t0=0.0,
            t1=1.0,
            dt0=None,
            y0=start,
            args=params,
            stepsize_controller=diffrax.PIDController(rtol=1e-5, atol=1e-5),
            saveat=diffrax.SaveAt(ts=jnp.asarray(times)),
        )
        return solution.ys, solution.stats["num_steps"]

    paths, steps = jax.jit(jax.vmap(integrate_one, in_axes=(None, 0)))(
        state.params, jnp.asarray(starts)
    )
    paths = np.asarray(paths)
    assert paths.shape == (len(starts), len(times), starts.shape[1])
    assert np.all(np.isfinite(paths))
    np.testing.assert_allclose(paths[:, 0], starts)
    # Saving the paths must give the same endpoints as standard prediction.
    np.testing.assert_allclose(
        paths[:, -1], model.solver.predict(starts, condition), rtol=1e-4, atol=1e-4,
    )
    return paths, np.asarray(steps)


cmp_trajectories = {}
cmp_solver_steps = {}
for name, model in cmp_models.items():
    cmp_trajectories[name], cmp_solver_steps[name] = integrate_comparison_paths(
        model, cmp_new_start_points, cmp_trajectory_times
    )
np.testing.assert_allclose(
    cmp_trajectories["RFM + biharmonic OT"], rfm_new_trajectories, rtol=1e-4, atol=1e-4
)
print(f"Integrated {len(cmp_new_start_points)} shared fresh source samples "
      f"at {len(cmp_trajectory_times)} saved times for each of {len(cmp_models)} models.")

In [ ]:
cmp_path_rows = []
fig = plt.figure(figsize=(22, 12), constrained_layout=True)
cmp_path_grid = fig.add_gridspec(2, 3)
cmp_path_axes = [fig.add_subplot(cmp_path_grid[row, column]) for row in range(2) for column in range(2)]
cmp_radius_ax = fig.add_subplot(cmp_path_grid[:, 2])
cmp_path_palette = plt.get_cmap("tab10")
cmp_path_extent = max(
    3.3, float(np.max(np.abs(np.concatenate(list(cmp_trajectories.values()))))) + 0.2
)
cmp_radius_ax.axhspan(inner_radius, outer_radius, color="tab:green", alpha=0.08, label="Annulus")
cmp_radius_ax.axhline(inner_radius, color="tab:red", linestyle="--")
cmp_radius_ax.axhline(outer_radius, color="gray", linestyle="--")
for ax, (name, trajectories) in zip(cmp_path_axes, cmp_trajectories.items()):
    ax.scatter(*rfm_source.T, s=8, alpha=0.15, color="tab:blue", label="Source cloud")
    ax.scatter(*rfm_target.T, s=8, alpha=0.15, color="tab:orange", label="Target cloud")
    ax.add_patch(plt.Circle((0, 0), inner_radius, fill=False, linestyle="--", color="tab:red"))
    ax.add_patch(plt.Circle((0, 0), outer_radius, fill=False, edgecolor="gray"))
    for path_index, path in enumerate(trajectories):
        color = cmp_path_palette(path_index % 10)
        ax.plot(*path.T, color=color, linewidth=2)
        ax.scatter(*path[0], s=60, color=color, edgecolor="white", zorder=5)
        ax.scatter(*path[-1], marker="X", s=80, color=color, edgecolor="white", zorder=6)
        for time_index in np.linspace(25, len(path) - 15, 4, dtype=int):
            ax.annotate(
                "", xy=path[time_index + 4], xytext=path[time_index],
                arrowprops={"arrowstyle": "->", "color": color, "lw": 1.6},
            )
    radii = np.linalg.norm(trajectories, axis=-1)
    in_annulus_mask = (radii >= inner_radius - 1e-4) & (radii <= outer_radius + 1e-4)
    segment_min_radii = straight_segment_min_radius(trajectories[:, :-1], trajectories[:, 1:])
    path_lengths = np.linalg.norm(np.diff(trajectories, axis=1), axis=-1).sum(axis=1)
    cmp_path_rows.append({
        "model": name,
        "paths_leaving_annulus": int(np.count_nonzero(~in_annulus_mask.all(axis=1))),
        "endpoints_in_annulus": float(np.mean(in_annulus_mask[:, -1])),
        "fraction_saved_points_in_annulus": float(np.mean(in_annulus_mask)),
        "min_sampled_radius": float(radii.min()),
        "max_sampled_radius": float(radii.max()),
        "mean_path_length": float(path_lengths.mean()),
        "max_step_length": float(np.linalg.norm(np.diff(trajectories, axis=1), axis=-1).max()),
        "max_segment_hole_penetration": float(np.maximum(inner_radius - segment_min_radii, 0.0).max()),
        "hole_crossing_segment_fraction": float(np.mean(segment_min_radii < inner_radius - 1e-3)),
        "max_annulus_violation": float(annulus_violation(trajectories).max()),
        "mean_adaptive_solver_steps": float(np.mean(cmp_solver_steps[name])),
    })
    ax.set(
        aspect="equal", xlabel="x", ylabel="y",
        xlim=(-cmp_path_extent, cmp_path_extent), ylim=(-cmp_path_extent, cmp_path_extent),
        title=f"{name}\n{cmp_path_rows[-1]['paths_leaving_annulus']}/{len(trajectories)} paths leave the annulus",
    )
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.2)
    cmp_radius_ax.plot(cmp_trajectory_times, radii.mean(axis=0), color=cmp_colors[name],
                       linewidth=2, label=f"{name} mean")
    cmp_radius_ax.fill_between(cmp_trajectory_times, radii.min(axis=0), radii.max(axis=0),
                               color=cmp_colors[name], alpha=0.12)
cmp_radius_ax.set(xlabel="Time t", ylabel="Radius ||x(t)||", xlim=(0, 1),
                  title="Mean radius; shaded range across the same paths")
cmp_radius_ax.legend(fontsize=8, loc="best")
cmp_radius_ax.grid(alpha=0.2)
fig.suptitle("Learned paths from the same fresh source samples; circles start, crosses end")
plt.show()

cmp_path_metrics = pd.DataFrame(cmp_path_rows).set_index("model")
display(cmp_path_metrics)

In [ ]:
# One table per model: endpoint distribution metrics on the fitted source cloud,
# path geometry on the shared fresh samples, and training cost.
cmp_summary = cmp_endpoint_metrics.loc[list(cmp_models), [
    "energy_distance_squared", "mean_error_to_target",
    "outside_annulus_fraction", "inside_hole_fraction", "mean_annulus_violation",
]].join(cmp_path_metrics[[
    "paths_leaving_annulus", "max_segment_hole_penetration", "mean_path_length",
]]).assign(training_seconds=pd.Series(cmp_training_seconds))
cmp_summary.index = pd.MultiIndex.from_tuples(
    [tuple(name.split(" + ")) for name in cmp_summary.index],
    names=["interpolation", "coupling"],
)
display(cmp_summary)

cmp_bar_metrics = [
    ("energy_distance_squared", "Squared energy distance to target"),
    ("outside_annulus_fraction", "Predicted endpoints outside annulus"),
    ("max_segment_hole_penetration", "Max path penetration into the hole"),
]
fig, axes = plt.subplots(1, len(cmp_bar_metrics), figsize=(18, 4.5), constrained_layout=True)
for ax, (column, title) in zip(axes, cmp_bar_metrics):
    table = cmp_summary[column].unstack("coupling").loc[["RFM", "FM"], ["biharmonic OT", "Euclidean OT"]]
    table.plot.bar(ax=ax, rot=0, color=["tab:green", "tab:gray"])
    ax.set(title=title, xlabel="Interpolation", ylabel=column)
    ax.legend(title="Coupling", fontsize=8)
    ax.grid(axis="y", alpha=0.2)
fig.suptitle("2 × 2 comparison: interpolation (x-axis) × coupling (bar color)")
plt.show()

### Reading the four-way comparison

Read the 2 × 2 table one factor at a time:

- **Coupling effect at fixed interpolation:** compare the two RFM rows with each other, then the two FM rows. Both couplings pair left points with right points. The coupling table shows whether the biharmonic cost changes that pairing, through `same_half_mass` and the biharmonic cost transported. `chord_through_hole_mass` shows how much coupled mass a straight Euclidean path would carry through the hole.
- **Interpolation effect at fixed coupling:** compare RFM against FM within each coupling column. The Euclidean models regress straight-line targets, and almost every left-to-right chord passes through the hole (`chord_through_hole_mass`). The RFM targets follow the annulus geometry during training, and their training interpolations are projected onto it.
- **Interaction:** a biharmonic coupling alone does not keep straight-line FM paths inside the annulus. Euclidean pairs can still be transported along geometry-aware RFM paths. The diagonal cells show whether the two ingredients help independently or only together.

All four `predict` calls integrate the learned field without projection. Endpoint violations therefore measure how well each learned field respects the domain, not a constraint enforced at inference. Metrics use the fitted clouds and the same six fresh samples. They describe this deterministic toy run, not generalization, so check them again whenever the kernel, OT regularization, interpolation resolution, or training length changes.